In [ ]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scipy.signal import spectrogram, butter, filtfilt, decimate
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm import tqdm

# Global Plotting Configuration
plt.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 20,
    'axes.labelsize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'figure.titlesize': 22
})

# Hardware Check
print("🧠 Available Devices:", tf.config.list_physical_devices())

In [ ]:
# Dataset & Signal Configuration
NUM_CHANNELS = 31
EXCLUDED_CHANNEL = 14
SAMPLE_RATE = 32000
ADC_ZERO = 32768
ELECTRODE_GAIN = 0.1042
TRIALS_PER_FILE = 180

DATASETS = {
    "0005": "Anestezie/M013_S001_SRCS3L_25,50,100_0005",
    "0006": "Anestezie/M013_S001_SRCS3L_25,50,100_0006",
    "0007": "Anestezie/M013_S001_SRCS3L_25,50,100_0007",
}

# Mapping: Dataset ID to Label Index
LABELS = {"0006": 0, "0005": 1, "0007": 2}  # 0: Deep, 1: Intermediate, 2: Light
STATE_DISPLAY_MAP = {
    "0006": "Deep",
    "0005": "Intermediate",
    "0007": "Light"
}
CLASS_NAMES = ["Deep", "Intermediate", "Light"]

In [ ]:
def apply_bandpass_filter(data, lowcut=0.5, highcut=300.0, fs=SAMPLE_RATE, order=3):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def get_channel_files(path):
    bin_files = sorted(
        [f for f in os.listdir(path) if f.endswith(".bin") and "-Ch" in f],
        key=lambda x: int(x.split("-Ch")[1].split(".")[0])
    )
    return [f for f in bin_files if int(f.split("-Ch")[1].split(".")[0]) != EXCLUDED_CHANNEL]

def compute_spectrogram(trial_data, fs=1000, nperseg=256, noverlap=128, mode='magnitude'):
    num_channels = trial_data.shape[0]
    spec_list = []
    for ch in range(num_channels):
        f, t, Sxx = spectrogram(trial_data[ch, :], fs=fs, nperseg=nperseg, noverlap=noverlap, mode=mode)
        spec_list.append(Sxx)
    return np.stack(spec_list, axis=-1)  # shape: (freq, time, channels)

def resize_spectrogram(spec, target_freq_bins=100, target_time_steps=1000):
    spec_resized = tf.image.resize(spec, [target_freq_bins, target_time_steps], method='bilinear')
    return spec_resized.numpy()

def normalize(X):
    mean = np.mean(X, axis=(0, 1, 2, 3), keepdims=True)
    std = np.std(X, axis=(0, 1, 2, 3), keepdims=True)
    return (X - mean) / (std + 1e-8)

In [ ]:
def process_single_trial(trial_idx, dataset_id, data_per_channel, samples_per_trial):
    start = trial_idx * samples_per_trial
    end = start + samples_per_trial

    downsample_factor = 32
    new_fs = SAMPLE_RATE // downsample_factor

    first_ch_raw = data_per_channel[0][start:end]
    first_ch_downsampled = decimate((first_ch_raw.astype(np.float32) - ADC_ZERO) * ELECTRODE_GAIN, downsample_factor)
    actual_length = len(first_ch_downsampled)

    trial_data = np.empty((NUM_CHANNELS, actual_length), dtype=np.float32)
    trial_data[0] = first_ch_downsampled

    for ch_idx in range(1, NUM_CHANNELS):
        ch_raw = data_per_channel[ch_idx][start:end]
        trial_data[ch_idx] = decimate((ch_raw.astype(np.float32) - ADC_ZERO) * ELECTRODE_GAIN, downsample_factor)[:actual_length]

    spec = compute_spectrogram(trial_data, fs=new_fs)
    spec = np.log1p(spec)
    spec_resized = resize_spectrogram(spec, 100, 1000)

    return spec_resized, LABELS[dataset_id]

def load_data():
    X, y = [], []
    trial_lengths = []

    for dataset_id, path in DATASETS.items():
        bin_files = get_channel_files(path)
        ch_data = np.memmap(os.path.join(path, bin_files[0]), dtype=np.int16, mode='r')
        trial_lengths.append(len(ch_data) // TRIALS_PER_FILE)

    samples_per_trial = min(trial_lengths)
    print(f"\n📏 Using fixed trial length: {samples_per_trial} samples per trial")

    for dataset_id, path in DATASETS.items():
        print(f"\n📂 Processing dataset {dataset_id}...")
        bin_files = get_channel_files(path)
        data_per_channel = [np.memmap(os.path.join(path, f), dtype=np.int16, mode='r') for f in bin_files]

        from joblib import Parallel, delayed
        results = Parallel(n_jobs=-1)(
            delayed(process_single_trial)(t_idx, dataset_id, data_per_channel, samples_per_trial)
            for t_idx in tqdm(range(TRIALS_PER_FILE), desc=f"Dataset {dataset_id}")
        )

        for spec, label in results:
            X.append(spec)
            y.append(label)

    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    print(f"\n✅ Final dataset shape: {X.shape}")
    return X, y

# Execuție încărcare
X, y = load_data()
X = normalize(X)

In [ ]:
def build_cnn_model(input_shape):
    inputs = Input(shape=input_shape)

    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.4)(x)

    x = GlobalAveragePooling2D()(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(len(LABELS), activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model_demo = build_cnn_model(X.shape[1:])
model_demo.summary()

In [ ]:
def train_model(X, y):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_accuracies = []
    fold_f1_scores = []
    best_model = None
    best_f1 = -1

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n🧪 Starting Fold {fold + 1} ------------------------------")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = build_cnn_model(X_train.shape[1:])

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=5, min_lr=1e-7)
        ]

        class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        class_weight_dict = dict(zip(np.unique(y_train), class_weights))

        history = model.fit(
            X_train, y_train,
            epochs=100,
            batch_size=16,
            validation_data=(X_test, y_test),
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=0
        )

        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        acc = np.mean(y_pred == y_test)
        f1 = f1_score(y_test, y_pred, average='weighted')

        fold_accuracies.append(acc)
        fold_f1_scores.append(f1)

        if f1 > best_f1:
            best_f1 = f1
            best_model = model

        print(f"📈 Fold {fold + 1} Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
        print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

        # Confusion Matrix Plot
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    annot_kws={"size": 16})
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"Confusion Matrix - Fold {fold + 1}")
        plt.tight_layout()
        plt.show()

    print(f"\n🏁 Mean Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
    print(f"🏁 Mean F1 Score: {np.mean(fold_f1_scores):.4f} ± {np.std(fold_f1_scores):.4f}")

    return best_model

# Rulează antrenamentul și salvează modelul
best_model = train_model(X, y)
best_model.save("cnn_model.h5")
print("✅ Model saved to cnn_model.h5")

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model([img_array])
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

def display_gradcam(img, heatmap, alpha=0.4):
    heatmap = np.uint8(255 * heatmap)
    jet = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    jet = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)
    jet = cv2.resize(jet, (img.shape[1], img.shape[0]))
    superimposed_img = jet * alpha + img * (1 - alpha)
    return np.clip(superimposed_img, 0, 255).astype(np.uint8)

def plot_anesthesia_evolution(X, y, model, samples_per_trial=25000):
    last_conv_layer = [layer.name for layer in model.layers if "conv2d" in layer.name][-1]
    duration_ms = (samples_per_trial / 32000) * 1000
    states = ["0006", "0005", "0007"]  # Deep, Intermediate, Light

    plt.figure(figsize=(16, 12))
    for i, state_code in enumerate(states):
        label_idx = LABELS[state_code]
        display_name = STATE_DISPLAY_MAP[state_code]
        
        idx = np.where(y == label_idx)[0][15]  
        sample = X[idx]

        heatmap = make_gradcam_heatmap(np.expand_dims(sample, axis=0), model, last_conv_layer)

        vis_img = np.mean(sample, axis=-1)
        vis_img = ((vis_img - vis_img.min()) / (vis_img.max() - vis_img.min() + 1e-8) * 255).astype(np.uint8)
        vis_img_rgb = np.stack([vis_img]*3, axis=-1)

        gradcam_res = display_gradcam(vis_img_rgb, heatmap, alpha=0.5)

        plt.subplot(3, 1, i+1)
        plt.imshow(gradcam_res, aspect='auto', origin='lower', extent=[0, duration_ms, 0, 500])
        plt.title(f"State: {display_name} Anesthesia", fontsize=18, fontweight='bold')
        plt.ylabel("Frequency (Hz)", fontsize=14)
        plt.xlabel("Time (ms)", fontsize=14)

    plt.tight_layout()
    plt.savefig("anesthesia_interpretability_comparison.png", dpi=300)
    plt.show()

# Rulează analiza Grad-CAM
plot_anesthesia_evolution(X, y, best_model)